## Notebook — Sync Divalto → Supabase (BE Budget par affaire)

### Objectif
Alimenter la table Supabase `be_divalto_mouvements` (mirror unifié) depuis
la table `lakehouse_gold.mouv_gold` pour permettre le suivi budget par
affaire dans l'app BE (rapprochement CCN/FCN/CFN/FFN).

### Source
- `lakehouse_gold.mouv_gold` — toutes les pièces avec préfixes Naskeo :
  - **CCN** : commande client Naskeo
  - **CFN** : commande fournisseur Naskeo (COGS sous-traitance)
  - **FCN** : facture client Naskeo
  - **FFN** : facture fournisseur Naskeo (COGS facturé)

### Destination
- `be_divalto_mouvements` — clé de dédup composite `(numero_piece, source)`
  - `source` = `'gescom'` (V1 — mouv_gold uniquement)

### Filtrage
On ne pousse que les lignes ayant un **code analytique** (`axe_0001 || axe_0002` non vide),
car c'est la clé de jointure avec `be_affaires.code_affaire`.

### Granularité
Une ligne par **(numero_piece, source)**. Les lignes brutes de `mouv_gold`
sont agrégées par pièce avec `SUM(mont)` et la convention `sens=2 → négatif`
(avoirs / contreparties). L'imputation analytique conservée est celle de la
première ligne (cas multi-imputé : à étendre si besoin).

### Pattern d'envoi
Edge Function `bulk-upsert` Supabase, mode `upsert` total, batches de 500.
Idempotent — relançable sans doublons.

### V2 à venir
- Croiser avec `C8_gold` pour récupérer le TTC compta (factures `source='compta'`)
- Lookup `axe_0001` via la commande liée pour les compta orphelines
- Filtrage sur les `code_affaire` connus de `be_affaires` (économie de bande passante)

In [ ]:
# ─────────────────────────────────────────
# 0) Credentials Supabase
# Variables injectées par le pipeline (Base parameters).
# Décommenter le bloc variableLibrary uniquement pour exécution manuelle.
# ─────────────────────────────────────────

# env = notebookutils.variableLibrary.getLibrary("env-temp")
# SUPABASE_URL       = env.SUPABASE_URL
# SYNC_SECRET        = env.SYNC_SECRET
# SUPABASE_ANON_KEY  = env.SUPABASE_PUBLISHABLE_KEY

import math, time, re, requests
from decimal import Decimal
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, DateType, IntegerType

GOLD_DB         = "lakehouse_gold"
BULK_UPSERT_URL = f"{SUPABASE_URL.rstrip('/')}/functions/v1/bulk-upsert"
BATCH_SIZE      = 500
SLEEP_S         = 0.02

# Préfixes Naskeo à pousser dans be_divalto_mouvements
PREFIX_CMD     = ["CCN", "CFN"]   # commandes (client + fournisseur)
PREFIX_FAC     = ["FCN", "FFN"]   # factures (client + fournisseur)
PREFIX_ALL_CMD = PREFIX_CMD
PREFIX_ALL_FAC = PREFIX_FAC

def bulk_headers():
    return {
        "Content-Type":   "application/json",
        "Authorization":  f"Bearer {SUPABASE_ANON_KEY}",
        "apikey":         SUPABASE_ANON_KEY,
        "x-sync-secret":  SYNC_SECRET,
    }

print("[0] Config OK")
print(f"    SUPABASE_URL    = {SUPABASE_URL}")
print(f"    SYNC_SECRET set = {bool(SYNC_SECRET)}")
print(f"    ANON_KEY set    = {bool(SUPABASE_ANON_KEY)}")
print(f"    Préfixes cmd    = {PREFIX_CMD}")
print(f"    Préfixes fac    = {PREFIX_FAC}")

In [ ]:
# ─────────────────────────────────────────
# 1) Helpers sérialisation JSON (identique au notebook IT)
# ─────────────────────────────────────────

_CTRL_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def _clean_str(s):
    if s is None:
        return None
    return _CTRL_RE.sub("", str(s)).rstrip() or None

def _to_jsonable(v):
    if v is None: return None
    if isinstance(v, bool): return v
    if isinstance(v, int): return v
    if isinstance(v, float):
        return None if (math.isnan(v) or math.isinf(v)) else round(v, 2)
    if isinstance(v, Decimal):
        try:
            f = float(v)
            return None if (math.isnan(f) or math.isinf(f)) else round(f, 2)
        except Exception:
            return _clean_str(str(v))
    if isinstance(v, (date, datetime)): return v.isoformat()
    if isinstance(v, (bytes, bytearray)):
        return _clean_str(v.decode("utf-8", errors="ignore"))
    if isinstance(v, str):
        s = _clean_str(v)
        if not s or s in ("1899-12-30", "0000-00-00"): return None
        if re.match(r"^\+?\d{5,}-\d{2}-\d{2}", s): return None
        return s
    if isinstance(v, dict): return {k: _to_jsonable(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)): return [_to_jsonable(x) for x in v]
    return _clean_str(str(v))

def _sanitize(rec):
    return {k: _to_jsonable(v) for k, v in rec.items()}

def _post_batch(table: str, conflict_key: str, records: list, on_conflict: str = "upsert"):
    if not records:
        return 0, None
    payload = {
        "table":        table,
        "conflict_key": conflict_key,
        "records":      [_sanitize(r) for r in records],
        "on_conflict":  on_conflict,
    }
    try:
        r = requests.post(BULK_UPSERT_URL, headers=bulk_headers(), json=payload, timeout=180)
        if r.status_code in (200, 201, 204):
            return len(records), None
        return 0, f"HTTP {r.status_code}: {r.text[:300]}"
    except Exception as e:
        return 0, str(e)

def send_to_supabase(table: str, conflict_key: str, records: list, label: str):
    ok_total, err_total, err_msgs = 0, 0, []
    total = len(records)
    for i in range(0, total, BATCH_SIZE):
        chunk = records[i:i + BATCH_SIZE]
        nb_ok, err = _post_batch(table, conflict_key, chunk)
        ok_total += nb_ok
        if err:
            err_total += len(chunk)
            err_msgs.append(f"batch {i//BATCH_SIZE + 1}: {err}")
        time.sleep(SLEEP_S)
    print(f"[{label}] ✅ {ok_total}/{total} envoyés | ❌ {err_total} erreurs")
    for m in err_msgs:
        print(f"  ⚠️  {m}")
    return ok_total, err_total

print("[1] Helpers OK")

In [ ]:
# ─────────────────────────────────────────
# 2) Lecture mouv_gold + filtre prérequis (axe analytique non vide)
# ─────────────────────────────────────────

df_mouv = spark.table(f"{GOLD_DB}.mouv_gold")
print(f"[2] mouv_gold chargé : {df_mouv.count()} lignes brutes")

# Pré-calcul du code_affaire (axe_0001 || axe_0002) pour filtrer en amont
df_mouv = df_mouv.withColumn(
    "_code_affaire",
    F.concat(
        F.coalesce(F.trim(F.col("axe_0001").cast(StringType())), F.lit("")),
        F.coalesce(F.trim(F.col("axe_0002").cast(StringType())), F.lit("")),
    ),
)

# On garde uniquement les pièces avec un code_affaire renseigné
# (sinon impossible de les rattacher à une affaire BE).
df_mouv = df_mouv.filter(F.col("_code_affaire") != "")
print(f"[2] Après filtre code_affaire non vide : {df_mouv.count()} lignes")

In [ ]:
# ─────────────────────────────────────────
# 3) Agrégation des COMMANDES (CCN + CFN)
# Granularité cible : 1 ligne par (FULLCDNO, source='gescom')
# ─────────────────────────────────────────

df_commandes = (
    df_mouv
    .filter(F.col("prefcdno").isin(PREFIX_CMD))
    .filter(F.col("FULLCDNO").isNotNull() & (F.trim(F.col("FULLCDNO")) != ""))
    .groupBy(
        F.trim(F.col("FULLCDNO").cast(StringType())).alias("numero_piece"),
        F.col("prefcdno").cast(StringType()).alias("prefpino"),
    )
    .agg(
        F.first(F.trim(F.col("axe_0001").cast(StringType())), ignorenulls=True).alias("axe_0001"),
        F.first(F.trim(F.col("axe_0002").cast(StringType())), ignorenulls=True).alias("axe_0002"),
        F.first(F.trim(F.col("tiers").cast(StringType())), ignorenulls=True).alias("tiers_code"),
        F.first(F.trim(F.col("tiers_fou").cast(StringType())), ignorenulls=True).alias("nom_tiers"),
        F.max(F.col("cddt").cast(DateType())).alias("date_piece"),
        F.max(F.year(F.col("cddt")).cast(IntegerType())).alias("exercice"),
        F.sum(
            F.when(F.col("sens") == 2, -F.col("mont"))
             .otherwise(F.col("mont"))
             .cast(DoubleType())
        ).alias("montant_ht"),
        F.concat_ws(" | ", F.collect_list(F.trim(F.col("des").cast(StringType())))).alias("libelle"),
    )
    .withColumn("type_mouv", F.col("prefpino"))
    .withColumn("source",    F.lit("gescom"))
    .withColumn("devise",    F.lit("EUR"))
)

nb_cmd = df_commandes.count()
print(f"[3] Commandes (CCN+CFN) : {nb_cmd} pièces")
df_commandes.select("numero_piece", "type_mouv", "axe_0001", "axe_0002", "montant_ht", "date_piece").show(5, truncate=False)

In [ ]:
# ─────────────────────────────────────────
# 4) Agrégation des FACTURES (FCN + FFN)
# Granularité cible : 1 ligne par (FULLFANO, source='gescom')
# ─────────────────────────────────────────

df_factures = (
    df_mouv
    .filter(F.col("preffano").isin(PREFIX_FAC))
    .filter(F.col("FULLFANO").isNotNull() & (F.trim(F.col("FULLFANO")) != ""))
    .groupBy(
        F.trim(F.col("FULLFANO").cast(StringType())).alias("numero_piece"),
        F.col("preffano").cast(StringType()).alias("prefpino"),
    )
    .agg(
        F.first(F.trim(F.col("axe_0001").cast(StringType())), ignorenulls=True).alias("axe_0001"),
        F.first(F.trim(F.col("axe_0002").cast(StringType())), ignorenulls=True).alias("axe_0002"),
        F.first(F.trim(F.col("tiers").cast(StringType())), ignorenulls=True).alias("tiers_code"),
        F.first(F.trim(F.col("tiers_fou").cast(StringType())), ignorenulls=True).alias("nom_tiers"),
        F.max(F.col("fadt").cast(DateType())).alias("date_piece"),
        F.max(F.year(F.col("fadt")).cast(IntegerType())).alias("exercice"),
        F.sum(
            F.when(F.col("sens") == 2, -F.col("mont"))
             .otherwise(F.col("mont"))
             .cast(DoubleType())
        ).alias("montant_ht"),
        F.concat_ws(" | ", F.collect_list(F.trim(F.col("des").cast(StringType())))).alias("libelle"),
    )
    .withColumn("type_mouv", F.col("prefpino"))
    .withColumn("source",    F.lit("gescom"))
    .withColumn("devise",    F.lit("EUR"))
)

nb_fac = df_factures.count()
print(f"[4] Factures (FCN+FFN) : {nb_fac} pièces")
df_factures.select("numero_piece", "type_mouv", "axe_0001", "axe_0002", "montant_ht", "date_piece").show(5, truncate=False)

In [ ]:
# ─────────────────────────────────────────
# 5) Union commandes + factures dans le schéma cible
# ─────────────────────────────────────────

TARGET_COLS = [
    "numero_piece", "prefpino", "type_mouv", "source",
    "axe_0001", "axe_0002",
    "date_piece", "exercice",
    "tiers_code", "nom_tiers", "libelle",
    "montant_ht", "devise",
]

df_all = (
    df_commandes.select(*TARGET_COLS)
    .unionByName(df_factures.select(*TARGET_COLS))
)

# Sécurité : on supprime les rares lignes sans code analytique cohérent
df_all = df_all.filter(
    (F.coalesce(F.col("axe_0001"), F.lit("")) != "") |
    (F.coalesce(F.col("axe_0002"), F.lit("")) != "")
)

nb_total = df_all.count()
print(f"[5] Total à pousser : {nb_total} pièces")
print("    Répartition par type_mouv :")
df_all.groupBy("type_mouv").count().orderBy("type_mouv").show()

In [ ]:
# ─────────────────────────────────────────
# 6) Envoi vers Supabase (be_divalto_mouvements)
# Conflict key composite : (numero_piece, source) → upsert idempotent
# ─────────────────────────────────────────

records = [row.asDict(recursive=True) for row in df_all.collect()]

ok, err = send_to_supabase(
    table        = "be_divalto_mouvements",
    conflict_key = "numero_piece,source",
    records      = records,
    label        = "BE_DIVALTO",
)

In [ ]:
# ─────────────────────────────────────────
# 7) Résumé final
# ─────────────────────────────────────────

print("")
print("=" * 60)
print("SYNC DIVALTO → SUPABASE BE BUDGET — RÉSUMÉ")
print("=" * 60)
print(f"  Commandes (CCN+CFN) lues  : {nb_cmd}")
print(f"  Factures  (FCN+FFN) lues  : {nb_fac}")
print(f"  Total pièces à pousser    : {nb_total}")
print(f"  Upsertées Supabase        : {ok}")
print(f"  Erreurs                   : {err}")
if err == 0:
    print("  ✅ Sync OK")
else:
    print("  ⚠️  Vérifier les logs ci-dessus")
print("=" * 60)